Import all libraries needed

In [1]:
#immport all important libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

import fiona
import geopandas as gpd
import geopy

import rasterio
from rasterio.features import shapes
from shapely.geometry import shape
from rasterio.merge import merge
import rioxarray

import glob
import os

opening Parcels Files and Cleaning

In [ ]:
gpk_path_new=r'file_path_Definite_V0_Parcels_analysis_07012026.gpkg'


In [ ]:
fiona.listlayers(gpk_path_new)
gdf_new = gpd.read_file(gpk_path_new, layer='parcels_boundaries_07042026')



In [ ]:
#drop unnecessary columns
gdf_new=gdf_new.drop(columns=['timestamp', 'begin', 'end',
       'altitudeMode', 'tessellate', 'extrude', 
       'visibility', 'drawOrder','icon','Season', 'Acquisition Channel','agronomist_id', 'Area'])


'"gdf_old=gdf_old.drop(columns=[\'timestamp\', \'begin\', \'end\',\n       \'altitudeMode\', \'tessellate\', \'extrude\',\n       \'visibility\', \'drawOrder\',\'icon\',\'Season\', \'Acquisition Channel\',\'agronomist_id\', \'Area\'])'

In [5]:
gdf_new.columns

Index(['id', 'Name', 'description', 'farmer_id', 'parcel_id', 'Full Name',
       'Status', 'Region', 'Province', 'agronomist', 'Crop Specie',
       'Seeding Date', 'geometry'],
      dtype='str')

In [ ]:
#ranaming columns
gdf_new.rename(columns={'Name': 'Region',
                        'description': 'Parcel_ID',
                        'farmer_id': 'farmer_name',
                        'parcel_id': 'parcel_name',
                        'Full Name': 'Status',
                        'Status': 'Province',
                        'Region': 'agronomist_id',
                        'agronomist': 'Crops',
                         'Seeding Date': 'Acquisition Channel',
                        'Province': 'Agronomist_Name',
                        'Crop Specie': 'Area'
                        
                        }, inplace=True)


"gdf_old.rename(columns={'Name': 'Region',\n                        'description': 'Parcel_ID',\n                        'farmer_id': 'farmer_name',\n                        'parcel_id': 'parcel_name',\n                        'Full Name': 'Status',\n                        'Status': 'Province',\n                        'Region': 'agronomist_id',\n                        'agronomist': 'Crops',\n                         'Seeding Date': 'Acquisition Channel',\n                        'Province': 'Agronomist_Name',\n                        'Crop Specie': 'Area'\n\n                        }, inplace=True)"

In [ ]:
#remove spaces in the names so channel attribution can take place
gdf_new['Agronomist_Name'] = gdf_new['Agronomist_Name'].str.replace(' ', '', regex=False)



Update channels

In [ ]:
"""there are various channels or farmer engagement which is not clean due to human errors,
by taking the names of the agronomists of the one of the channels, 
we will correct them all to their respective channel 
"""

In [ ]:
CHannel_1_AGRONOMISTS = [
    'Agronomist_1',
    'Agronomist_2',
    'Agronomist_3',
    'Agronomist_4',
    'Agronomist_5',
    'Agronomist_6',
]

In [ ]:
def assign_channel(Agronomist_Name):
    if pd.isna(Agronomist_Name):
        return "Channel_0"
    elif Agronomist_Name in CHannel_1_AGRONOMISTS:
        return "channel_1"
    elif Agronomist_Name == "T_Prod":
        return "Channel_2"
    else:
        return "channel_0"

#exucuting the function
gdf_new['Acquisition Channel'] = gdf_new['Agronomist_Name'].apply(assign_channel)

In [11]:
gdf_new.columns

Index(['id', 'Region', 'Parcel_ID', 'farmer_name', 'parcel_name', 'Status',
       'Province', 'agronomist_id', 'Agronomist_Name', 'Crops', 'Area',
       'Acquisition Channel', 'geometry'],
      dtype='str')

In [12]:
gdf_new.head(2)

,id,Region,Parcel_ID,farmer_name,parcel_name,Status,Province,agronomist_id,Agronomist_Name,Crops,Area,Acquisition Channel,geometry
0,boundaries_nigeria_07042026.1,Niger,2aab91bb-39c6-49d6-b164-f462472401e1,Mohammed Babalagan,Mohammed Babalagan-P1,monitoring,Agaie,87a16176-29fc-4c06-a3bc-e14f7d11312b,UmarBABALAGAN,Groundnut,1.297434150581306,Indirect-OCPA,"POLYGON ((432200.334 556211.135, 432016.033 55..."
1,boundaries_nigeria_07042026.2,Niger,bce95fd7-4221-4c28-b106-73c6058b6df8,Mohammed Babalagan,Mohammed Babalagan-P2,monitoring,Agaie,87a16176-29fc-4c06-a3bc-e14f7d11312b,UmarBABALAGAN,Maize,0.9723429547474302,Indirect-OCPA,"POLYGON ((430807.593 560196.106, 430804.268 56..."


Open Excel sheet to be joined with geopackage

In [ ]:
"""""
the excel have extra informations such as farmers id that is needed for further analysis
"""

In [ ]:
df1 = pd.read_excel(r'file_path_All_Parcels_All_13042026.xlsx',
                    sheet_name='Sheet1')

In [14]:
df1.columns

Index(['Agronomist', 'Farmer', 'name', 'Region', 'Province', 'Status',
       'Area (ha)', 'Aquisition Channel', 'parcel_id', 'farmer_id', 'Link',
       'Total_Engaged_Area', 'Farmer_Eligibility'],
      dtype='str')

In [ ]:
#joinin the parcel excel file to the goepack file
gdf_new= gdf_new.merge(df1, left_on='Parcel_ID', right_on='parcel_id', how='left')



In [ ]:
#coverte area to float
gdf_new['Area'] = pd.to_numeric(gdf_new['Area'],errors='coerce')




Creating a Unique Farmers ID and thier total engage areas

In [ ]:
#take famers id when availabe, else creat a whole new based on his other informations
gdf_new['group_key'] = np.where(
    gdf_new['farmer_id'].notna(),
    gdf_new['farmer_id'].astype(str),
    (
        gdf_new['farmer_name'].astype(str) + '_' +
        gdf_new['agronomist_id'].astype(str) + '_' +
        gdf_new['Province_x'].astype(str)
    )
)


# Agregat all the parcels areas of every farmers into his total engaged area
gdf_new['Total_Engaged_Area'] = (gdf_new.groupby('group_key')['Area'].transform('sum'))

"gdf_old['group_key'] = np.where(\n    gdf_old['farmer_id'].notna(),\n    gdf_old['farmer_id'].astype(str),\n    (\n        gdf_old['farmer_name'].astype(str) + '_' +\n        gdf_old['agronomist_id'].astype(str) + '_' +\n        gdf_old['Province_x'].astype(str)\n    )\n)\n\ngdf_old['Total_Engaged_Area'] = (gdf_old.groupby('group_key')['Area'].transform('sum'))"

In [18]:
gdf_new['Total_Engaged_Area'].isna().sum()

np.int64(0)

Who are the eligible farmers?

In [ ]:
    """
    these are regions of smallholder who sometimes have even less than 0.1 ha, but for
    operational efficiency, 
    an eligible farmer is a farmer with a total engeged area of at least 0.6 Ha
    """

In [ ]:
#defining the funtions
def Eligible_Engaged(Total_Engaged_Area):
    if pd.isna(Total_Engaged_Area):
        return None
    elif Total_Engaged_Area <0.6:
        return "NO"
    else:
        return "YES"
    
#exectuting the function
gdf_new['Farmer_Eligibility'] = gdf_new['Total_Engaged_Area'].apply(Eligible_Engaged)


In [ ]:
#know the count of parcels owned by every farmer
gdf_new['Number_of_Parcels'] = gdf_new.groupby('group_key')['Parcel_ID'].transform('count')


In [21]:
gdf_new.columns

Index(['id', 'Region_x', 'Parcel_ID', 'farmer_name', 'parcel_name', 'Status_x',
       'Province_x', 'agronomist_id', 'Agronomist_Name', 'Crops', 'Area',
       'Acquisition Channel', 'geometry', 'Agronomist', 'Farmer', 'name',
       'Region_y', 'Province_y', 'Status_y', 'Area (ha)', 'Aquisition Channel',
       'parcel_id', 'farmer_id', 'Link', 'Total_Engaged_Area',
       'Farmer_Eligibility', 'group_key', 'Number_of_Parcels'],
      dtype='str')

Define parcels on fallow from parcels without crops

In [2]:

"""some parcels are on fallow while others do not just have crops, with the ids of the parcels
 on fallow we can feed it to their crops columns
 """

'some parcels are on fallow while others do not just have crops, with the ids of the parcels\n on fallow we can feed it to their crops columns\n '

In [ ]:
fallow_parcels = [
    'fallow_parcel_ID_1',
    'fallow_parcel_ID_2',
    'fallow_parcel_ID_3',
    'fallow_parcel_ID_5',
    'fallow_parcel_ID_45',
    'fallow_parcel_ID_6',
    'fallow_parcel_ID_7',
    ....................
    'fallow_parcel_ID_50',
    'fallow_parcel_ID_51',
]



In [23]:

fallow = (
    gdf_new['Crops'].isna() &
    gdf_new['Parcel_ID'].isin(fallow_parcels)
)

gdf_new.loc[fallow, 'Crops'] = 'Fallow'


In [ ]:
#see the number of parcels that do not have crops
gdf_new['Crops'].isna().sum()

np.int64(837)

In [25]:
gdf_new['group_key'].nunique()

32969

In [ ]:
#this is to streamline all the farmers with total engaged area less than 0.6 as ineligible
gdf_new_yes=gdf_new[gdf_new['Farmer_Eligibility']=='YES']
print(gdf_new_yes['group_key'].nunique())


"gdf_new_yes=gdf_new[gdf_new['Farmer_Eligibility']=='YES']\nprint(gdf_new_yes['group_key'].nunique())\n"

In [ ]:
#refining columns
gdf_new.drop(columns=['Agronomist_Name', 'Region_y', 'Province_y', 'Status_y', 'Area (ha)', 'Aquisition Channel',
       'parcel_id', 'farmer_id', 'Farmer_Eligibility', 'Farmer', 'name'], inplace=True)

In [31]:
gdf_new.rename(columns={'Region_x': 'Region',
                        'Province_x': 'Province',
                        'Status_x': 'Status',
                        'agronomist_id': 'Agronomist_ID'}, inplace=True)

In [32]:
gdf_new.columns

Index(['id', 'Region', 'Parcel_ID', 'farmer_name', 'parcel_name', 'Status',
       'Province', 'Agronomist_ID', 'Crops', 'Area', 'Acquisition Channel',
       'geometry', 'Agronomist', 'Link', 'Total_Engaged_Area', 'group_key',
       'Number_of_Parcels'],
      dtype='str')

In [33]:
gdf_new['Crops'].value_counts()

Crops
Maize             13072
Sorghum            7889
Soybean            7696
Groundnut          6264
Pearl Millet       5150
Cowpea             1501
Wheat               122
Sunflower            99
Sesame               91
Forage Legum'e       19
Fallow               13
Melon                 2
Name: count, dtype: int64

In [38]:
print('the number of unique farmers is:', gdf_new['group_key'].nunique())
print('the number of unique parcels is:', gdf_new['Parcel_ID'].nunique())
print('the total area is:', gdf_new['Area'].sum())

the number of unique farmers is: 32969
the number of unique parcels is: 42755
the total area is: 103960.93447003703


Loading raster files

In [ ]:

def load_raster(raster_path):
    with rasterio.open(raster_path) as src:
        data = src.read(1)
        transform = src.transform
        crs = src.crs
        nodata = src.nodata
    return data, transform, crs, nodata


raster_file = r"file_path_\raster.tif"

data, transform, crs, nodata = load_raster(raster_file)

In [ ]:
    """
    the raster file is a lancover file from esri sentinel-2 with codes corresponding to each class
    we need to print them out some we can rerun the classifications
    """

In [ ]:

codes = np.unique(data)

print(codes)



[ 0  1  2  4  5  7  8 11]


In [44]:
# Get unique class codes from the landcover dataand map them to class names
class_map = {
    1: "Water",
    2: "Forest",
    4: "flooded vegetation",
    5: "Crop land",
    7:"Builded Area",
    8:"Bare Soil",
    11: "Rangelands"
    
}


In [45]:
# Get unique class codes from the landcover dataand map them to class names
unique_codes = np.unique(data)
labels = [class_map.get(code, "Unknown") for code in unique_codes]

pd.DataFrame({
    "Code": unique_codes,
    "Class Name": labels
})


,Code,Class Name
0,0,Unknown
1,1,Water
2,2,Forest
3,4,flooded vegetation
4,5,Crop land
5,7,Builded Area
6,8,Bare Soil
7,11,Rangelands


Vectorisation of raster

In [46]:

data = data.astype(np.int32)

records = []

mask = data != nodata if nodata is not None else None

for geom, value in shapes(data, mask=mask, transform=transform):
    if value == 0:
        continue

    records.append({
        "geometry": shape(geom),
        "code": int(value),
        "class_name": class_map.get(int(value), "Unknown")
    })

gdf_raster = gpd.GeoDataFrame(records, crs=crs)

In [47]:
len(gdf_raster)

548134

Treating the raster for the Constraints zones

In [48]:
#filter only shapes of interest

gdf_no_farming = gdf_raster[gdf_raster["class_name"].isin(["Water", "Forest", "flooded vegetation", "Builded Area", "Bare Soil"])]


In [49]:
gdf_no_farming.is_valid.value_counts()

True    124898
Name: count, dtype: int64

In [50]:
invalid = ~gdf_no_farming.is_valid
gdf_no_farming.loc[invalid, "geometry"] = gdf_no_farming.loc[invalid, "geometry"].buffer(0)

In [ ]:
gdf_no_farming = gdf_no_farming.dissolve(by="class_name")

In [ ]:
gdf_no_farming.head(5)

In [ ]:
gdf_constraints=gdf_no_farming

Import open source Protected areas

In [ ]:
"""
the protected areas are zones protected for their wild life ans biodiverity,

they are be protect from all sorte of disturbance, the reasone for which farms in this areas 
cannot be considered

"""


In [ ]:
protected = gpd.read_file(r"file_path\Protected_Areas.gpkg")

# Ensure same CRS
protected = protected.to_crs(gdf_constraints.crs)

# merge the constraints and protected areas into a single GeoDataFrame

all_constraints = gpd.GeoDataFrame(
    pd.concat([gdf_constraints, protected], ignore_index=True),
    crs=gdf_constraints.crs
)

In [ ]:
#Add all Constraints areas
constraint_union = gdf_constraints.geometry.union_all()

In [ ]:
#classify parcels based on their relationship to the constraint areas
parcels = gdf_new
parcels = parcels.to_crs(gdf_constraints.crs)

def classify(parcel_geom):
    if not parcel_geom.intersects(constraint_union):
        return "completely eligible"
    elif parcel_geom.within(constraint_union):
        return "ineligible"
    else:
        return "partially eligible"

parcels["eligibility"] = parcels.geometry.apply(classify)

In [ ]:
parcels['eligibility'].value_counts()

In [ ]:
#export parcels into a geopackage for viewing in Qgis
parcels.to_file(r'file_path\parcel_eligibility.gpkg',layer='Corrected_Boundaries', driver='GPKG')
